# Python 工程示例

预计阅读时间：15-20 分钟。

本节以 GLM 为例，使用 OpenAI 兼容 API，演示一个完整的 Prompt 工程示例：让大模型生成一段符合规范的算法代码，并以 JSON 返回结构化结果。

## 任务目标

我们要做一个“算法代码生成助手”。它接收一个算法需求，然后返回一个 JSON 对象，包含：

1. `title`：算法标题；
2. `mermaid`：用于展示算法流程的 Mermaid 代码；
3. `code`：完整 Python 代码；
4. `notes`：简短说明或检查提示。

JSON 比纯 Markdown 更适合工程代码处理：程序可以稳定读取字段，再按业务需要展示、保存或测试。

## 准备客户端

上一节已经介绍了依赖安装、API Key 保存和 OpenAI 兼容 API 的基本用法。这里直接使用同样的方法读取 `ZAI_API_KEY` 并创建客户端。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("ZAI_API_KEY")
if not api_key:
    raise RuntimeError("请先设置环境变量 ZAI_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://open.bigmodel.cn/api/paas/v4/",
)

MODEL_NAME = "glm-5.2"  # 示例模型名，以平台当前可用模型为准

## 设计算法需求

先把用户需求写成一个普通字符串。这里以“斐波那契数列”为例，因为它足够简单，同时能体现输入、输出、边界条件和循环过程。

In [2]:
algorithm_task = """请实现斐波那契数列算法。
函数名为 fibonacci。
输入为非负整数 n。
返回第 n 个斐波那契数，其中 fibonacci(0) = 0，fibonacci(1) = 1。
如果 n 小于 0，抛出 ValueError。"""

print(algorithm_task)

请实现斐波那契数列算法。
函数名为 fibonacci。
输入为非负整数 n。
返回第 n 个斐波那契数，其中 fibonacci(0) = 0，fibonacci(1) = 1。
如果 n 小于 0，抛出 ValueError。


## 构造 prompt

这里的重点不是“让模型写代码”，而是把返回字段和格式写清楚。把 prompt 封装成函数后，每次只需要传入新的算法需求即可。

In [3]:
SYSTEM_PROMPT = """你是一个严谨的 Python 编程助教。
你生成的代码应当简洁、可读，并适合初学者学习。"""


def build_algorithm_prompt(task: str) -> str:
    """Build a prompt for generating algorithm code as JSON.

    Args:
        task: A natural language description of the algorithm requirement.

    Returns:
        A complete user prompt with JSON field constraints.
    """
    return f"""请根据下面的需求生成算法代码。

需求：
{task}

返回要求：
1. 只返回一个合法 JSON 对象，不要使用 Markdown 代码块包裹；
2. JSON 必须包含字段：title、mermaid、code、notes；
3. title 是字符串，表示算法名称；
4. mermaid 是字符串，只包含 Mermaid flowchart 代码，不要包含 ```mermaid；
5. mermaid 节点文本必须使用双引号包裹，例如 A["检查 n 是否小于 0"]、B{{"n 小于 0?"}}；
6. mermaid 节点文本中不要直接写 fibonacci(0)、fibonacci(1)、range(2, n + 1) 这类包含括号或加号的代码表达式；
7. code 是字符串，只包含完整 Python 代码，不要包含 ```python；
8. Python 代码必须采用 Google 风格 docstring；
9. 函数的 Args、Returns 和 Raises 必须清楚说明输入、输出和异常；
10. notes 是字符串列表，用于说明边界条件和人工检查重点。
"""


user_prompt = build_algorithm_prompt(algorithm_task)
print(user_prompt)

请根据下面的需求生成算法代码。

需求：
请实现斐波那契数列算法。
函数名为 fibonacci。
输入为非负整数 n。
返回第 n 个斐波那契数，其中 fibonacci(0) = 0，fibonacci(1) = 1。
如果 n 小于 0，抛出 ValueError。

返回要求：
1. 只返回一个合法 JSON 对象，不要使用 Markdown 代码块包裹；
2. JSON 必须包含字段：title、mermaid、code、notes；
3. title 是字符串，表示算法名称；
4. mermaid 是字符串，只包含 Mermaid flowchart 代码，不要包含 ```mermaid；
5. mermaid 节点文本必须使用双引号包裹，例如 A["检查 n 是否小于 0"]、B{"n 小于 0?"}；
6. mermaid 节点文本中不要直接写 fibonacci(0)、fibonacci(1)、range(2, n + 1) 这类包含括号或加号的代码表达式；
7. code 是字符串，只包含完整 Python 代码，不要包含 ```python；
8. Python 代码必须采用 Google 风格 docstring；
9. 函数的 Args、Returns 和 Raises 必须清楚说明输入、输出和异常；
10. notes 是字符串列表，用于说明边界条件和人工检查重点。



## 调用模型

下面代码会把 prompt 发送给 GLM。这里要求模型返回 JSON，因此后续代码会先用 `json.loads` 解析，再交给业务展示逻辑。

In [4]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0.2,
)

result_text = response.choices[0].message.content
print(result_text)

{
  "title": "斐波那契数列算法",
  "mermaid": "flowchart TD\n    A[\"开始\"] --> B{\"n 小于 0?\"}\n    B -- \"是\" --> C[\"抛出 ValueError\"]\n    B -- \"否\" --> D{\"n 等于 0?\"}\n    D -- \"是\" --> E[\"返回 0\"]\n    D -- \"否\" --> F{\"n 等于 1?\"}\n    F -- \"是\" --> G[\"返回 1\"]\n    F -- \"否\" --> H[\"初始化前两个数\"]\n    H --> I[\"循环计算下一个数\"]\n    I --> J{\"是否到达第 n 个数?\"}\n    J -- \"否\" --> I\n    J -- \"是\" --> K[\"返回当前数\"]",
  "code": "def fibonacci(n):\n    \"\"\"计算第 n 个斐波那契数。\n\n    Args:\n        n (int): 非负整数，表示要计算的斐波那契数列的项数。\n\n    Returns:\n        int: 第 n 个斐波那契数。\n\n    Raises:\n        ValueError: 如果 n 小于 0。\n    \"\"\"\n    if n < 0:\n        raise ValueError(\"n 必须是非负整数\")\n    if n == 0:\n        return 0\n    if n == 1:\n        return 1\n    \n    a, b = 0, 1\n    for _ in range(2, n + 1):\n        a, b = b, a + b\n    return b",
  "notes": [
    "边界条件：当 n 为 0 时返回 0，当 n 为 1 时返回 1。",
    "异常处理：当输入 n 小于 0 时，必须抛出 ValueError 异常。",
    "人工检查重点：验证 n 为 0、1 和较大正整数时的返回值是否正确，以及负数输入是否正确抛出异常。"
  ]
}


## 业务展示

假设业务侧需要把模型结果展示给用户。模型返回 JSON 后，业务代码先解析字段，再组装成 Markdown 渲染。这样既保留结构化数据，也能获得适合阅读的展示效果。

In [5]:
import json
from IPython.display import Markdown, display

result = json.loads(result_text)

markdown_text = f"""# {result["title"]}

## 算法流程

```mermaid
{result["mermaid"]}
```

## Python 代码

```python
{result["code"]}
```

## 说明

{chr(10).join(f"- {note}" for note in result["notes"])}
"""

display(Markdown(markdown_text))

# 斐波那契数列算法

## 算法流程

```mermaid
flowchart TD
    A["开始"] --> B{"n 小于 0?"}
    B -- "是" --> C["抛出 ValueError"]
    B -- "否" --> D{"n 等于 0?"}
    D -- "是" --> E["返回 0"]
    D -- "否" --> F{"n 等于 1?"}
    F -- "是" --> G["返回 1"]
    F -- "否" --> H["初始化前两个数"]
    H --> I["循环计算下一个数"]
    I --> J{"是否到达第 n 个数?"}
    J -- "否" --> I
    J -- "是" --> K["返回当前数"]
```

## Python 代码

```python
def fibonacci(n):
    """计算第 n 个斐波那契数。

    Args:
        n (int): 非负整数，表示要计算的斐波那契数列的项数。

    Returns:
        int: 第 n 个斐波那契数。

    Raises:
        ValueError: 如果 n 小于 0。
    """
    if n < 0:
        raise ValueError("n 必须是非负整数")
    if n == 0:
        return 0
    if n == 1:
        return 1
    
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
```

## 说明

- 边界条件：当 n 为 0 时返回 0，当 n 为 1 时返回 1。
- 异常处理：当输入 n 小于 0 时，必须抛出 ValueError 异常。
- 人工检查重点：验证 n 为 0、1 和较大正整数时的返回值是否正确，以及负数输入是否正确抛出异常。


## 检查输出

拿到模型结果后，不要直接复制到项目中。至少检查：

- 返回内容是否能被 `json.loads` 正确解析；
- JSON 是否包含 `title`、`mermaid`、`code`、`notes` 四个字段；
- Mermaid 是否能渲染；
- Mermaid 节点文本是否使用双引号包裹，避免 `fibonacci(0)`、`range(2, n + 1)` 等表达式导致解析错误；
- Python 代码是否完整、可运行；
- 函数名、参数和返回值是否符合需求；
- docstring 是否采用 Google 风格；
- 边界情况是否合理，例如 `n=0`、`n=1`、`n<0`。

这正是后续 Coding Agent 工作流的基础：模型负责生成候选方案，人负责审查、运行和决定是否采纳。

## 练习

把 `algorithm_task` 改成“实现阶乘函数”或“实现括号匹配检查”，然后重新运行 `build_algorithm_prompt(algorithm_task)` 和 API 调用，观察模型是否仍能稳定返回符合字段要求的 JSON。